### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
for chunk in model.stream("Why do parrots talk?"):
    print(chunk.text, end="|", flush=True)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||Par|rots| don|’t| “|talk|”| in| the| same| way| humans| do|—| they| **|m|im|ic|**| sounds| they| hear|,| especially| human| speech|.| | The| ability| to| copy| words| and| phrases| is| a| by|‑|product| of| several| biological| and| behavioral| traits| that| evolved| for| very| different| reasons|.| Here|’s| a| breakdown| of| why| parro|ts| are| such| good| imit|ators|:

|---

|##| |1|.| The| anatomy| of| a| par|rot|’s| voice| box|

||| Feature| || What| it| does| || Why| it| matters| for| mimic|ry| |
|||---------|||--------------|||----------------|-----------||
||| **|S|yr|inx|**| (|the| bird| equivalent| of| a| l|ary|nx|)| || Located| at| the| base| of| the| tr|ache|a|,| the| syr|inx| has| two| sets| of| vibrating| membranes| that| can| be| controlled| independently|.| || Allows| a| single| bird| to| produce| two| different| sounds| at| the| same| time| (|e|.g|.,| a| whistle| plus| a| trill|),| giving| parro|ts| an| enormous

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using the function. Use function call.', 'tool_calls': [{'id': 'fc_d951615d-b5c4-46b6-b182-13e3da3487aa', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 127, 'total_tokens': 170, 'completion_time': 0.089828867, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005135136, 'prompt_tokens_details': None, 'queue_time': 0.286839056, 'total_time': 0.094964003}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a02ab2-3460-7b92-ab1c-064cb5ad628c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_d951615d-b5c4-46b6-b182-13e3da3487aa', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata

### Tool Execution Loops

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print("Executing tool:", tool_call)
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

Executing tool: {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_48293758-f1ec-44e0-9d63-e1df5c717fbf', 'type': 'tool_call'}
It's sunny in Boston.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "What\'s the weather in Boston?" We need to fetch weather via function get_weather. Use function.', 'tool_calls': [{'id': 'fc_48293758-f1ec-44e0-9d63-e1df5c717fbf', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 126, 'total_tokens': 176, 'completion_time': 0.108226066, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.004716545, 'prompt_tokens_details': None, 'queue_time': 0.313241624, 'total_time': 0.112942611}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a02acc-9531-7ad0-bcb7-2c86e3a9c82f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bost